# Inside the Game: load the synthetic dataset into the lakehouse

Loads the five JSON Lines files written by `python -m insidegame.datasets.generate` into Delta tables with
explicit schemas, adds the derived columns the semantic model needs, and builds a `clubs` dimension.

**Prerequisites**
- This notebook has a default lakehouse attached (Lakehouse explorer, *Add lakehouse*).
- `matches.jsonl`, `events.jsonl`, `players.jsonl`, `moments.jsonl` and `tracking.jsonl` are uploaded to
  `Files/insidegame/` in that lakehouse.

Re-running is safe: every table is overwritten. A file that does not match its schema stops the run (FAILFAST).

In [ ]:
SOURCE_DIR = "Files/insidegame"
PHASE_MINUTES = 15
PHASE_LABELS = ["00-15", "15-30", "30-45", "45-60", "60-75", "75-90+"]
READ_MODE = "FAILFAST"
TACTICAL_PATTERN = "tactical_pattern"
OUTPUT_TABLES = ["matches", "clubs", "events", "players", "moments", "tracking"]

# Spark DDL per source file; kept equal to insidegame.fabric.schema.spark_ddl by the offline check.
SOURCE_SCHEMAS = {
    "matches": (
        "match_id STRING, seed BIGINT, competition STRING, venue STRING, home STRING, "
        "away STRING, home_team_id STRING, away_team_id STRING, home_style STRING, "
        "away_style STRING, home_goals BIGINT, away_goals BIGINT, events BIGINT, "
        "home_possession_pct DOUBLE, home_xg DOUBLE, away_xg DOUBLE, key_moments BIGINT"
    ),
    "events": (
        "event_id STRING, match_id STRING, sequence BIGINT, period BIGINT, match_second DOUBLE, "
        "minute BIGINT, event_type STRING, team_id STRING, player_id STRING, "
        "possession_id BIGINT, outcome STRING, receiver_id STRING, opponent_id STRING, "
        "under_pressure BOOLEAN, length_m DOUBLE, ball_speed_kmh DOUBLE, shot_result STRING, "
        "seed BIGINT, side STRING, minute_label STRING, start_x DOUBLE, start_y DOUBLE, "
        "end_x DOUBLE, end_y DOUBLE, score_home BIGINT, score_away BIGINT, xg DOUBLE, "
        "pass_difficulty DOUBLE, momentum DOUBLE, rhythm_state STRING, xt_added DOUBLE, "
        "moment_kinds ARRAY<STRING>"
    ),
    "players": (
        "match_id STRING, passes_attempted BIGINT, passes_completed BIGINT, "
        "progressive_passes BIGINT, final_third_entries BIGINT, shots BIGINT, "
        "shots_on_target BIGINT, goals BIGINT, xg DOUBLE, max_shot_speed_kmh DOUBLE, "
        "carries BIGINT, carry_distance_m DOUBLE, tackles BIGINT, interceptions BIGINT, "
        "pressures BIGINT, fouls BIGINT, saves BIGINT, xt_added DOUBLE, "
        "line_breaking_passes BIGINT, opponents_bypassed BIGINT, player_id STRING, "
        "team_id STRING, name STRING, pass_accuracy DOUBLE, avg_pass_difficulty DOUBLE, "
        "position STRING, shirt_number BIGINT"
    ),
    "moments": (
        "match_id STRING, moment_id STRING, kind STRING, period BIGINT, match_second DOUBLE, "
        "minute BIGINT, minute_label STRING, importance DOUBLE, team_id STRING, "
        "player_ids ARRAY<STRING>, event_ids ARRAY<STRING>, facts STRING"
    ),
    "tracking": (
        "match_id STRING, event_id STRING, sequence BIGINT, period BIGINT, match_second DOUBLE, "
        "ball_x DOUBLE, ball_y DOUBLE, players ARRAY<ARRAY<STRING>>, sprint_player_id STRING, "
        "sprint_kmh DOUBLE"
    ),
}

In [ ]:
def read_source(name):
    """Read one JSON Lines file with its explicit schema, failing on any malformed record."""
    return spark.read.schema(SOURCE_SCHEMAS[name]).option("mode", READ_MODE).json(f"{SOURCE_DIR}/{name}.jsonl")


sources = {name: read_source(name) for name in SOURCE_SCHEMAS}
for name, frame in sources.items():
    print(name, frame.count())

In [ ]:
from pyspark.sql import functions as F

last_phase = F.lit(len(PHASE_LABELS) - 1)
phase_index = F.greatest(F.lit(0), F.least(F.floor((F.col("minute") - 1) / PHASE_MINUTES), last_phase))
phase_labels = F.array(*[F.lit(label) for label in PHASE_LABELS])

events = (
    sources["events"]
    .withColumn("phase", F.element_at(phase_labels, (phase_index + 1).cast("int")))
    .withColumn("moment_kinds", F.to_json("moment_kinds"))
)

moments = (
    sources["moments"]
    .withColumn(
        "scout_pattern",
        F.when(F.col("kind") == TACTICAL_PATTERN, F.get_json_object("facts", "$.pattern")),
    )
    .withColumn("player_ids", F.to_json("player_ids"))
    .withColumn("event_ids", F.to_json("event_ids"))
)

tracking = sources["tracking"].withColumn("players", F.to_json("players"))

matches = sources["matches"]


def club_side(side):
    """Select one side's club id, name, and style from the matches table."""
    return matches.select(
        F.col(f"{side}_team_id").alias("team_id"), F.col(side).alias("club_name"), F.col(f"{side}_style").alias("style")
    )


clubs = club_side("home").unionByName(club_side("away")).distinct()

tables = {
    "matches": matches,
    "clubs": clubs,
    "events": events,
    "players": sources["players"],
    "moments": moments,
    "tracking": tracking,
}

In [ ]:
def save(frame, name):
    """Overwrite one Delta table (schema included) in the default lakehouse."""
    frame.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print("wrote", name)


save(tables["matches"], "matches")
save(tables["clubs"], "clubs")
save(tables["events"], "events")
save(tables["players"], "players")
save(tables["moments"], "moments")
save(tables["tracking"], "tracking")

In [ ]:
for name in OUTPUT_TABLES:
    print(f"{name}: {spark.table(name).count()} rows")